# Teledyne LeCroy T3PS30051P DC power supply

This example adapts the single-output DC supply workflow to QCoDeS. The T3PS30051P supports 0-30 V and a 0-5 A current limit.

You need a connected instrument and a working VISA backend. Replace the resource address below with your instrument's address. Check the wiring and choose limits appropriate for the connected device before enabling output. Construction does not reset or disable an already enabled output. Closing the connection does not disable it either.

The example keeps the output disabled unless you explicitly uncomment the output-enable command.

This model does not respond to `*IDN?`. The driver supplies the known vendor and model locally, with no serial number or firmware information, so initialization and station snapshots do not send identification queries. Queries wait one second before reading a response. Close any existing connection from another notebook or application before connecting here.

In [1]:
from qcodes.instrument_drivers.teledyne import TeledyneT3PS30051P
from qcodes.station import Station

In [ ]:
resource_name = "USB0::0xF4EC::0x1410::T0103C21190153::0::INSTR"
pwr = TeledyneT3PS30051P("pwr", resource_name)
station = Station(pwr)

KeyboardInterrupt: 

## Configure voltage and current

Turn the output off before changing the setpoints. QCoDeS parameters validate values before sending commands. Unlike the original standalone driver, getters return numeric values instead of just printing them.

In [ ]:
pwr.output(False)
pwr.voltage(10.0)
pwr.current(0.4)
print(f"Voltage setpoint: {pwr.voltage()} V")
print(f"Current limit: {pwr.current()} A")

## Inspect status and control output

`output` reads status bit 4, while `mode` reports constant-voltage (`CV`) or constant-current (`CC`) operation. `status` returns the complete status register as an integer.

Uncomment the output-enable command only after checking that the voltage and current limit are appropriate.

In [ ]:
print(f"Status register: {pwr.status():#06x}")
print(f"Output enabled: {pwr.output()}")
print(f"Regulation mode: {pwr.mode()}")

# pwr.output(True)

## Read the actual output

Measurements are separate from programmed setpoints. A station snapshot records both settings and measurements.

In [ ]:
print(f"Measured voltage: {pwr.measured_voltage()} V")
print(f"Measured current: {pwr.measured_current()} A")
print(f"Measured power: {pwr.measured_power()} W")
snapshot = station.snapshot(update="All")

## Disable output and close

Run this cleanup cell even if a preceding cell raises an error. For unattended scripts, put output shutdown and connection cleanup in a `try`/`finally` block.

In [ ]:
try:
    pwr.output(False)
finally:
    pwr.close()